In [2]:
import sys
from pathlib import Path
import pandas as pd

import folium
HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

In [7]:
from server.scripts.region_grid.h3_9 import h3_children, h3_boundary_latlon, h3_center

seed_path = Path("../adaptive_hexsearch/map/seed_ledger.csv").resolve()
df_seed = pd.read_csv(seed_path)

TARGET_RES = 9
seed_col = "tile_id"
res_col = "h3_res"

h3_res9_cells = []
for _, row in df_seed[[seed_col, res_col]].dropna().iterrows():
    h3_res9_cells.extend(h3_children(str(row[seed_col]), TARGET_RES))

h3_res9_cells = sorted(set(h3_res9_cells))
df_h3_9 = pd.DataFrame({"h3_9": h3_res9_cells})

# Build polygons and centers for mapping.
df_h3_9["boundary"] = df_h3_9["h3_9"].map(h3_boundary_latlon)
df_h3_9[["center_lat", "center_lon"]] = pd.DataFrame(
    df_h3_9["h3_9"].map(h3_center).tolist(),
    index=df_h3_9.index,
)
df_h3_9 = df_h3_9.reset_index(drop=True).rename(columns={"h3_9": "tileID"})

df_h3_9.to_csv("h3_9.csv", index=False)
print(f"Loaded seed rows: {len(df_seed)}")
print(f"Unique H3 res {TARGET_RES} cells: {len(df_h3_9)}")
display(df_h3_9.head())

Loaded seed rows: 599
Unique H3 res 9 cells: 4193


,tileID,boundary,center_lat,center_lon
0,89194ad0003ffff,"[[51.46014332971915, -0.04859045289981933], [5...",51.458396,-0.048160
1,89194ad0007ffff,"[[51.45762777059396, -0.045639633896778734], [...",51.455880,-0.045209
2,89194ad000bffff,"[[51.459931894144546, -0.05320053551237643], [...",51.458184,-0.052770
3,89194ad000fffff,"[[51.45741639871367, -0.0502494073392683], [51...",51.455669,-0.049819
4,89194ad0013ffff,"[[51.46287013250036, -0.04693136398562476], [5...",51.461123,-0.046501


In [3]:
map_center = [df_h3_9["center_lat"].mean(), df_h3_9["center_lon"].mean()]
m = folium.Map(location=map_center, zoom_start=12, tiles="CartoDB positron")

for row in df_h3_9.itertuples(index=False):
    folium.Polygon(
        locations=row.boundary,
        color="#0f766e",
        weight=1,
        fill=True,
        fill_color="#14b8a6",
        fill_opacity=0.18,
        tooltip=f"h3_9: {row.h3_9}",
    ).add_to(m)

out_path = Path("../adaptive_hexsearch/map/h3_9_seed_map.html").resolve()
m.save(str(out_path))
print(f"Map saved to: {out_path}")

Map saved to: C:\Users\kylec\OneDrive\Desktop\react_project\london-explorer\server\scripts\adaptive_hexsearch\map\h3_9_seed_map.html
